# Text Mining — Final Solution
**Spring Semester 2025/2026 | NOVA IMS**

**Model:** FinBERT fine-tuned end-to-end (`ProsusAI/finbert`)  
**Task:** 3-class tweet sentiment — Bearish (0) · Bullish (1) · Neutral (2)  
**Pipeline:** Minimal text cleaning → HuggingFace Trainer → predict on test.csv → pred_xx.csv

> Run all cells top-to-bottom. No external variables needed.
> GPU strongly recommended (CUDA). CPU works but is slower (~1h for training).

## 1. Installs & Imports

In [ ]:
!pip install transformers torch datasets accelerate -q
!pip install nltk -q

In [ ]:
import re, warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset
from sklearn.metrics import (
    classification_report, accuracy_score, f1_score
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch {torch.__version__} | Transformers ready')

## 2. Load Data

In [ ]:
df_train = pd.read_csv('train.csv')
df_test  = pd.read_csv('test.csv')

print(f'Train: {df_train.shape}  |  Test: {df_test.shape}')
print(f'Train columns: {list(df_train.columns)}')
print(f'Test  columns: {list(df_test.columns)}')
print(f'\nClass distribution (train):')
print(df_train['label'].value_counts().sort_index()
      .rename({0:'Bearish',1:'Bullish',2:'Neutral'}))

## 3. Text Preprocessing

Minimal cleaning for transformer models: remove URLs and normalise whitespace.  
The FinBERT tokeniser handles subword splitting, punctuation, and casing internally.
Aggressive cleaning (lowercasing, stopword removal, lemmatisation) would destroy
information the model was pre-trained to use.

In [ ]:
def preprocess_transformer(text_list):
    """
    Minimal cleaning for FinBERT:
      - Remove URLs (no semantic value)
      - Collapse extra whitespace
    Everything else (casing, punctuation, @mentions, $tickers) is preserved
    because FinBERT was pre-trained on financial text including these patterns.
    """
    results = []
    for text in tqdm(text_list, desc='Preprocessing'):
        text = str(text)
        text = re.sub(r'http\S+|www\S+', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        results.append(text)
    return results


print('Preprocessing train...')
X_train_raw = preprocess_transformer(df_train['text'].tolist())
y_train     = df_train['label'].tolist()

print('Preprocessing test...')
X_test_raw  = preprocess_transformer(df_test['text'].tolist())

print(f'\nExample after preprocessing:')
print(f'  Original : {df_train["text"].iloc[0]}')
print(f'  Cleaned  : {X_train_raw[0]}')

## 4. Tokenisation

FinBERT uses a WordPiece tokeniser with a finance-domain vocabulary.  
`max_length=128` covers >99% of tweets without truncation.
All sequences are padded to the same length within each batch.

In [ ]:
MODEL_NAME   = 'ProsusAI/finbert'
MAX_LENGTH   = 128

print(f'Loading tokeniser: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH,
    )

# Build HuggingFace Dataset objects
# Train on the FULL training set (all 9543 tweets) for best test performance
train_hf = Dataset.from_dict({'text': X_train_raw, 'label': y_train})
test_hf  = Dataset.from_dict({'text': X_test_raw})

print('Tokenising train...')
train_hf = train_hf.map(tokenize_fn, batched=True)

print('Tokenising test...')
test_hf  = test_hf.map(tokenize_fn, batched=True)

print(f'\nTrain size : {len(train_hf)}')
print(f'Test size  : {len(test_hf)}')
print(f'Features   : {train_hf.features}')

## 5. Model — FinBERT Fine-tuned

**Why FinBERT?**  
FinBERT (`ProsusAI/finbert`) is a BERT encoder pre-trained on 4.9B tokens of financial
text (Reuters, Bloomberg, financial news). It already understands financial vocabulary
like tickers, earnings, bear/bull markets — domain knowledge that general BERT lacks.
Fine-tuning adds a 3-class classification head on top.

**Architecture:**  
`[CLS] tweet tokens [SEP]` → 12 transformer layers → [CLS] embedding (768d) → Linear(768→3) → softmax

In [ ]:
print(f'Loading model: {MODEL_NAME}')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={0: 'Bearish', 1: 'Bullish', 2: 'Neutral'},
    label2id={'Bearish': 0, 'Bullish': 1, 'Neutral': 2},
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Model architecture  : {model.config.model_type}')

## 6. Fine-tuning

Training on the full training set (no held-out validation fold here — this is the
final model, not the experimental one). Hyperparameters tuned in `tm_tests_xx`:
3 epochs, batch 16, warmup 10%, weight decay 0.01.

In [ ]:
training_args = TrainingArguments(
    output_dir               = './finbert_final',
    num_train_epochs         = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    warmup_ratio             = 0.1,
    weight_decay             = 0.01,
    logging_steps            = 50,
    save_strategy            = 'no',      # no checkpoint — full train, no eval
    seed                     = 42,
    fp16                     = torch.cuda.is_available(),  # mixed precision if GPU
    report_to                = 'none',
)

trainer = Trainer(
    model        = model,
    args         = training_args,
    train_dataset= train_hf,
)

print('Starting fine-tuning...')
print(f'  Epochs          : {training_args.num_train_epochs}')
print(f'  Batch size      : {training_args.per_device_train_batch_size}')
print(f'  Training samples: {len(train_hf)}')
print(f'  Mixed precision : {training_args.fp16}')
print()
trainer.train()
print('\nFine-tuning complete.')

## 7. Predict on test.csv

Run the fine-tuned model on the 299 test tweets and produce `pred_xx.csv`.

In [ ]:
print('Generating predictions on test set...')
preds_output = trainer.predict(test_hf)
final_preds  = preds_output.predictions.argmax(axis=1)

label_map = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}
pred_series = pd.Series(final_preds)
print(f'\nPrediction distribution:')
for label, cnt in pred_series.value_counts().sort_index().items():
    print(f'  {label_map[label]} ({label}): {cnt} tweets ({cnt/len(final_preds)*100:.1f}%)')

## 8. Save Submission File

In [ ]:
submission = pd.DataFrame({
    'id'   : df_test['id'],
    'label': final_preds,
})

submission.to_csv('pred_xx.csv', index=False)
print('Saved: pred_xx.csv')
print(f'Rows: {len(submission)}')
print(submission.head(10).to_string(index=False))